# 11F — 2025–2026 Forward Performance

This notebook evaluates the already-frozen 2024 critical-minerals signals over later returns. It first audits the latest date available through this WRDS account. It does **not** reconstruct filings, LLM labels, embeddings, or propagated scores.

The primary source is CRSP. Yahoo Finance is deliberately not used automatically because ticker changes, acquisitions, and delistings can create survivorship and linkage errors.

## 1. Imports, paths, and frozen design

The portfolio definitions and timing conventions are declared before later returns are opened.

In [1]:
from pathlib import Path
from getpass import getpass
import hashlib, json, math
import numpy as np
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
if HERE.name=='quarterly_sec_pilot': PILOT_ROOT=HERE; ROOT=HERE.parents[1]
elif HERE.name=='geopolitical_exposure': PILOT_ROOT=HERE/'quarterly_sec_pilot'; ROOT=HERE.parent
elif HERE.name=='Asset_Embeddings': ROOT=HERE; PILOT_ROOT=ROOT/'geopolitical_exposure'/'quarterly_sec_pilot'
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve(); PILOT_ROOT=ROOT/'geopolitical_exposure'/'quarterly_sec_pilot'
SOURCE=PILOT_ROOT/'three_seed_specifications'
EVAL=PILOT_ROOT/'quarterly_portfolio_evaluation'
OUT=PILOT_ROOT/'forward_performance_2025_2026'
RAW=OUT/'data'/'raw'; PROCESSED=OUT/'data'/'processed'; RESULTS=OUT/'results'; MANIFESTS=OUT/'manifests'
for p in [RAW,PROCESSED,RESULTS,MANIFESTS]: p.mkdir(parents=True,exist_ok=True)
def sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
VINTAGES=['2024q1','2024q2','2024q3','2024q4']
WINDOWS={'2024q1':['2024-06-30','2024-07-31','2024-08-31'],
         '2024q2':['2024-09-30','2024-10-31','2024-11-30'],
         '2024q3':['2024-12-31','2025-01-31','2025-02-28'],
         '2024q4':['2025-04-30','2025-05-31','2025-06-30']}
LONG_START={'2024q1':'2024-06-01','2024q2':'2024-09-01','2024q3':'2024-12-01','2024q4':'2025-04-01'}
TAIL=0.10
print('Project root:',ROOT)

Project root: /Users/harmandeep/Desktop/Projects/Asset_Embeddings


## 2. Verify and load the frozen signals

Later returns are still unopened here. We preserve the 2024 text seeds and 11C embedding scores exactly as previously created.

In [2]:
design=json.loads((SOURCE/'manifests'/'three_seed_specification_design.json').read_text())
assert design['outcomes_accessed'] is False
scores_path=SOURCE/'results'/'three_specification_propagated_scores.parquet'
seeds_path=SOURCE/'processed'/'automated_quarterly_seed_panel.parquet'
scores=pd.read_parquet(scores_path)
seeds=pd.read_parquet(seeds_path)
signals=scores[(scores.channel=='critical_minerals_overall') & (scores.target=='signed_score')].copy()
signals['permco']=pd.to_numeric(signals.permco,errors='raise').astype('int64')
seeds=seeds[seeds.channel.eq('critical_minerals_overall')].copy()
seeds['firm_id']=pd.to_numeric(seeds.firm_id,errors='raise').astype('int64')
required_permcos=sorted(set(signals.permco) | set(seeds.firm_id))
frozen=pd.DataFrame([{'score_sha256':sha256(scores_path),'seed_sha256':sha256(seeds_path),
 'score_rows':len(signals),'seed_rows':len(seeds),'required_permcos':len(required_permcos),
 'returns_accessed':False}])
frozen.to_parquet(MANIFESTS/'frozen_signal_audit.parquet',index=False)
display(frozen)

,score_sha256,seed_sha256,score_rows,seed_rows,required_permcos,returns_accessed
0,f0fab057c8772d01ea4022bc1878db7c39d8a799d48745...,8a2cfbb6883463c2d2282c27d0f144881766fd9599f2d3...,35712,174,1699,False


## 3. Connect to WRDS

This cell requests credentials only when a live connection is not already present. The password is not printed or stored.

In [6]:
import wrds
connection_ok=False
if 'db' in globals() and getattr(db,'connection',None) is not None:
    try:
        db.raw_sql('select 1 as connection_test')
        connection_ok=True
    except Exception:
        try: db.close()
        except Exception: pass
if not connection_ok:
    username=input('WRDS username: ').strip()
    password=getpass('WRDS password: ')
    db=wrds.Connection(wrds_username=username,wrds_password=password)
    del password
    db.raw_sql('select 1 as connection_test')
print('WRDS connection is active.')

Loading library list...
Done
WRDS connection is active.


## 4. Audit the actual CRSP cutoff available to this account

The audit probes both legacy and CIZ-style monthly files. A successful maximum-date query establishes the account-specific cutoff; filenames and WRDS website dates are not treated as proof of query access.

In [7]:
CANDIDATES=[('crsp_a_stock','msf_v2'),('crsp_a_stock','msf'),('crsp','msf')]
audit=[]; source_specs=[]
for schema,table in CANDIDATES:
    qualified=f'{schema}.{table}'
    try:
        columns=db.raw_sql("select column_name from information_schema.columns where table_schema=%(schema)s and table_name=%(table)s order by ordinal_position",params={'schema':schema,'table':table}).column_name.str.lower().tolist()
        date_col=next((c for c in ['mthcaldt','date'] if c in columns),None)
        return_col=next((c for c in ['mthret','ret'] if c in columns),None)
        if not date_col or not return_col or 'permno' not in columns or 'permco' not in columns:
            raise KeyError('required monthly fields absent')
        latest=pd.to_datetime(db.raw_sql(f'select max({date_col}) as latest_date from {qualified}').latest_date.iloc[0])
        spec={'qualified_table':qualified,'date_col':date_col,'return_col':return_col,
              'dlret_col':'dlret' if 'dlret' in columns else None,
              'price_col':next((c for c in ['mthprc','prc'] if c in columns),None),
              'shares_col':next((c for c in ['shrout'] if c in columns),None),
              'latest_date':latest}
        source_specs.append(spec)
        audit.append({'table':qualified,'queryable':True,'latest_date':latest,'error':None})
    except Exception as error:
        try: db.connection.rollback()
        except Exception: pass
        audit.append({'table':qualified,'queryable':False,'latest_date':pd.NaT,'error':type(error).__name__})
cutoff_audit=pd.DataFrame(audit)
cutoff_audit.to_parquet(MANIFESTS/'wrds_crsp_cutoff_audit.parquet',index=False)
display(cutoff_audit)
if not source_specs: raise RuntimeError('No CRSP monthly table is queryable; inspect the audit before proceeding.')
source_spec=max(source_specs,key=lambda x:x['latest_date'])
CRSP_CUTOFF=source_spec['latest_date']
print('Selected source:',source_spec['qualified_table'])
print('Latest CRSP monthly date accessible to this account:',CRSP_CUTOFF.date())
if CRSP_CUTOFF < pd.Timestamp('2025-06-30'):
    print('WARNING: the available cutoff cannot complete both Q3 and Q4 three-month windows.')

,table,queryable,latest_date,error
0,crsp_a_stock.msf_v2,True,2025-12-31,None
1,crsp_a_stock.msf,True,2024-12-31,None
2,crsp.msf,True,2024-12-31,None


Selected source: crsp_a_stock.msf_v2
Latest CRSP monthly date accessible to this account: 2025-12-31


## 5. Download only the required 2025–2026 CRSP observations

Queries are restricted to firms in the frozen experiment and are checkpointed by calendar year. For legacy CRSP, ordinary and delisting returns are compounded. Where several securities map to one PERMCO, the largest-market-equity security represents the firm-month; the audit reports all such selections.

In [8]:
def download_year(year,spec,batch_size=350):
    path=RAW/f"crsp_required_firms_{year}.parquet"
    if path.exists(): return pd.read_parquet(path), 'checkpoint'
    end=min(pd.Timestamp(f'{year}-12-31'),spec['latest_date'])
    if end < pd.Timestamp(f'{year}-01-01'): return pd.DataFrame(), 'unavailable'
    extras=[]
    if spec['dlret_col']: extras.append(f", {spec['dlret_col']} as dlret")
    if spec['price_col']: extras.append(f", {spec['price_col']} as price")
    if spec['shares_col']: extras.append(f", {spec['shares_col']} as shares_outstanding")
    frames=[]
    for start in range(0,len(required_permcos),batch_size):
        ids=','.join(map(str,required_permcos[start:start+batch_size]))
        sql=(f"select permno, permco, {spec['date_col']} as date, {spec['return_col']} as ret"+''.join(extras)+
             f" from {spec['qualified_table']} where permco in ({ids}) and {spec['date_col']} between '{year}-01-01' and '{end.date()}'")
        frames.append(db.raw_sql(sql,date_cols=['date']))
        print(f'{year}: batch {start//batch_size+1}/{math.ceil(len(required_permcos)/batch_size)}')
    raw=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    raw.to_parquet(path,index=False)
    return raw,'downloaded'
parts=[]; acquisition=[]
for year in range(2025,CRSP_CUTOFF.year+1):
    part,status=download_year(year,source_spec)
    if len(part): parts.append(part)
    acquisition.append({'year':year,'status':status,'rows':len(part),
                        'first_date':pd.to_datetime(part.date).min() if len(part) else pd.NaT,
                        'last_date':pd.to_datetime(part.date).max() if len(part) else pd.NaT})
acquisition=pd.DataFrame(acquisition); acquisition.to_parquet(MANIFESTS/'return_acquisition_audit.parquet',index=False)
display(acquisition)

2025: batch 1/5
2025: batch 2/5
2025: batch 3/5
2025: batch 4/5
2025: batch 5/5


,year,status,rows,first_date,last_date
0,2025,downloaded,19936,2025-01-31,2025-12-31


## 6. Construct firm-month total returns and join the 2024 checkpoint

The code preserves firms with a −100% delisting return. Missing returns are not converted to zero.

In [9]:
new=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=['permno','permco','date','ret'])
for c in ['permno','permco','ret','dlret','price','shares_outstanding']:
    if c in new: new[c]=pd.to_numeric(new[c],errors='coerce')
new['date']=pd.to_datetime(new.date)
if 'dlret' in new:
    new['total_return']=(1+new.ret.fillna(0))*(1+new.dlret.fillna(0))-1
    new.loc[new.ret.isna() & new.dlret.isna(),'total_return']=np.nan
else: new['total_return']=new.ret
if {'price','shares_outstanding'}.issubset(new): new['market_equity']=new.price.abs()*new.shares_outstanding
else: new['market_equity']=np.nan
new=new.sort_values(['date','permco','market_equity','permno'],ascending=[True,True,False,True])
new_firm=new.drop_duplicates(['date','permco'])[['permco','date','total_return','permno','market_equity']].copy()
new_firm=new_firm.rename(columns={'date':'eom','total_return':'eom_ret'})
local_path=ROOT/'active_passive_extension'/'measures'/'firm_returns_v2_total_return'/'monthly_firm_returns_2024.parquet'
local=pd.read_parquet(local_path)[['permco','eom','eom_ret']]; local['eom']=pd.to_datetime(local.eom)
monthly=pd.concat([local,new_firm[['permco','eom','eom_ret']]],ignore_index=True)
monthly['permco']=pd.to_numeric(monthly.permco,errors='raise').astype('int64')
monthly['eom_ret']=pd.to_numeric(monthly.eom_ret,errors='coerce')
monthly=monthly.sort_values(['permco','eom']).drop_duplicates(['permco','eom'],keep='last')
monthly.to_parquet(PROCESSED/'crsp_firm_monthly_through_cutoff.parquet',index=False)
display(pd.DataFrame([{'firms':monthly.permco.nunique(),'firm_months':len(monthly),
 'first_date':monthly.eom.min(),'last_date':monthly.eom.max(),'missing_returns':monthly.eom_ret.isna().sum()}]))

,firms,firm_months,first_date,last_date,missing_returns
0,5202,77694,2024-01-31,2025-12-31,0


## 7. Freeze portfolio membership

For each vintage, embedding model, and seed design, we retain three portfolio constructions: direct LLM seeds, embedding-only discoveries after excluding every seed, and their union. Positive and negative legs are defined separately.

In [10]:
members=[]
for (vintage,model,spec),g in signals.groupby(['vintage','embedding_model','seed_specification']):
    own=set(seeds[(seeds.vintage==vintage)&(seeds.seed_specification==spec)].firm_id)
    all_seed=set(seeds[seeds.vintage==vintage].firm_id)
    sg=(seeds[(seeds.vintage==vintage)&(seeds.seed_specification==spec)]
        .groupby('firm_id',as_index=False).signed_score.mean())
    for r in sg.itertuples():
        if r.signed_score!=0: members.append({'vintage':vintage,'embedding_model':model,'seed_specification':spec,'portfolio':'llm_only','permco':int(r.firm_id),'leg':'positive' if r.signed_score>0 else 'negative','signal':r.signed_score})
    candidates=g[~g.permco.isin(all_seed)].drop_duplicates('permco').copy()
    low=candidates.propagated_score.quantile(TAIL); high=candidates.propagated_score.quantile(1-TAIL)
    eg=candidates[(candidates.propagated_score<=low)|(candidates.propagated_score>=high)]
    for r in eg.itertuples():
        leg='positive' if r.propagated_score>=high else 'negative'
        members.append({'vintage':vintage,'embedding_model':model,'seed_specification':spec,'portfolio':'embedding_only','permco':r.permco,'leg':leg,'signal':r.propagated_score})
membership=pd.DataFrame(members).drop_duplicates(['vintage','embedding_model','seed_specification','portfolio','permco','leg'])
combined=pd.concat([membership[membership.portfolio=='llm_only'],membership[membership.portfolio=='embedding_only']],ignore_index=True)
combined['portfolio']='llm_plus_embedding'
membership=pd.concat([membership,combined],ignore_index=True).drop_duplicates(['vintage','embedding_model','seed_specification','portfolio','permco','leg'])
membership.to_parquet(RESULTS/'frozen_portfolio_membership.parquet',index=False)
display(membership.groupby(['portfolio','seed_specification','leg'],as_index=False).permco.nunique())

,portfolio,seed_specification,leg,permco
0,embedding_only,broad,negative,541
1,embedding_only,broad,positive,601
2,embedding_only,combined,negative,348
3,embedding_only,combined,positive,543
4,embedding_only,sector_enriched,negative,309
5,embedding_only,sector_enriched,positive,624
6,llm_only,broad,negative,4
7,llm_only,broad,positive,2
8,llm_only,combined,negative,15
9,llm_only,combined,positive,5


## 8. Evaluate immediate and through-cutoff performance

Returns are equal weighted across available constituents each month. The immediate horizon uses the predeclared three-month windows. The persistence horizon runs from the vintage-specific investable start through the CRSP cutoff. Results include coverage so disappearing firms cannot be silently ignored.

In [11]:
panel=membership.merge(monthly,on='permco',how='left',validate='many_to_many')
panel['month_end']=panel.eom+pd.offsets.MonthEnd(0)
rows=[]; monthly_rows=[]
for keys,g in panel.groupby(['vintage','embedding_model','seed_specification','portfolio','leg']):
    vintage,model,spec,portfolio,leg=keys
    declared=pd.to_datetime(WINDOWS[vintage])
    for horizon,mask in [('immediate_3m',g.month_end.isin(declared)),
                         ('through_cutoff',(g.eom>=pd.Timestamp(LONG_START[vintage]))&(g.eom<=CRSP_CUTOFF))]:
        x=g[mask].dropna(subset=['eom_ret']).copy()
        mr=x.groupby('month_end',as_index=False).agg(portfolio_return=('eom_ret','mean'),observed_firms=('permco','nunique'))
        mr['vintage']=vintage;mr['embedding_model']=model;mr['seed_specification']=spec;mr['portfolio']=portfolio;mr['leg']=leg;mr['horizon']=horizon
        monthly_rows.append(mr)
        rows.append({'vintage':vintage,'embedding_model':model,'seed_specification':spec,'portfolio':portfolio,'leg':leg,'horizon':horizon,
                     'selected_firms':g.permco.nunique(),'observed_firms':x.permco.nunique(),'months':len(mr),
                     'cumulative_return':np.prod(1+mr.portfolio_return)-1 if len(mr) else np.nan,
                     'mean_monthly_return':mr.portfolio_return.mean() if len(mr) else np.nan,
                     'monthly_volatility':mr.portfolio_return.std(ddof=1) if len(mr)>1 else np.nan})
leg_performance=pd.DataFrame(rows); portfolio_monthly=pd.concat(monthly_rows,ignore_index=True)
keys=['vintage','embedding_model','seed_specification','portfolio','horizon']
wide=leg_performance.pivot(index=keys,columns='leg',values='cumulative_return').reset_index()
wide['positive_minus_negative']=wide.get('positive',np.nan)-wide.get('negative',np.nan)
wide=wide.merge(leg_performance.groupby(keys,as_index=False).agg(months=('months','min'),minimum_firms=('observed_firms','min')),on=keys)
leg_performance.to_parquet(RESULTS/'portfolio_leg_performance.parquet',index=False)
portfolio_monthly.to_parquet(RESULTS/'portfolio_monthly_returns.parquet',index=False)
wide.to_parquet(RESULTS/'long_short_performance.parquet',index=False)
display(wide.sort_values(['horizon','vintage','embedding_model','seed_specification','portfolio']))

,vintage,embedding_model,seed_specification,portfolio,horizon,negative,positive,positive_minus_negative,months,minimum_firms
0,2024q1,rs_binary,broad,embedding_only,immediate_3m,0.062229,0.071740,0.009511,3,147
2,2024q1,rs_binary,broad,llm_only,immediate_3m,-0.074608,-0.132554,-0.057946,3,1
4,2024q1,rs_binary,broad,llm_plus_embedding,immediate_3m,0.058540,0.070268,0.011729,3,148
6,2024q1,rs_binary,combined,embedding_only,immediate_3m,0.051839,0.072590,0.020751,3,146
8,2024q1,rs_binary,combined,llm_only,immediate_3m,-0.068882,-0.110657,-0.041775,3,3
...,...,...,...,...,...,...,...,...,...,...
135,2024q4,rs_l_min,combined,llm_only,through_cutoff,0.523173,0.611751,0.088578,9,5
137,2024q4,rs_l_min,combined,llm_plus_embedding,through_cutoff,0.156062,0.165830,0.009768,9,147
139,2024q4,rs_l_min,sector_enriched,embedding_only,through_cutoff,0.083681,0.265284,0.181603,9,141
141,2024q4,rs_l_min,sector_enriched,llm_only,through_cutoff,0.546366,0.609256,0.062890,9,3


## 9. Coverage gates and reproducibility manifest

A horizon is complete only when every declared month is available. Partial 2026 performance is retained but explicitly labelled as such.

In [12]:
coverage=[]
available_months=set(monthly.eom.dt.to_period('M'))
for vintage,dates in WINDOWS.items():
    required=set(pd.to_datetime(dates).to_period('M'))
    coverage.append({'vintage':vintage,'required_months':len(required),'available_months':len(required&available_months),
                     'complete':required.issubset(available_months),'missing_months':','.join(map(str,sorted(required-available_months)))})
coverage=pd.DataFrame(coverage);coverage.to_parquet(MANIFESTS/'forward_window_coverage.parquet',index=False)
artifacts=[]
for path in sorted(RESULTS.glob('*.parquet')):
    artifacts.append({'artifact':path.name,'rows':len(pd.read_parquet(path)),'bytes':path.stat().st_size,'sha256':sha256(path)})
inventory=pd.DataFrame(artifacts);inventory.to_parquet(MANIFESTS/'output_inventory.parquet',index=False)
manifest={'stage':'2025_2026_forward_performance','crsp_source':source_spec['qualified_table'],
 'crsp_cutoff':str(CRSP_CUTOFF.date()),'tail_fraction':TAIL,'windows':WINDOWS,'long_start':LONG_START,
 'portfolio_weighting':'equal_weighted_monthly','llm_and_embedding_signals_reestimated':False,
 'yfinance_used':False,'score_sha256':sha256(scores_path),'seed_sha256':sha256(seeds_path),
 'artifacts':inventory.to_dict('records')}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
(MANIFESTS/'forward_performance_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
display(coverage);display(inventory)
print('Actual accessible CRSP cutoff:',CRSP_CUTOFF.date())
print('Complete immediate windows:',int(coverage.complete.sum()),'of',len(coverage))
print('Manifest:',manifest['sha256'])

,vintage,required_months,available_months,complete,missing_months
0,2024q1,3,3,True,
1,2024q2,3,3,True,
2,2024q3,3,3,True,
3,2024q4,3,3,True,


,artifact,rows,bytes,sha256
0,frozen_portfolio_membership.parquet,14720,113751,b7ca6134ed8680c0ffef230d4f034e33de5d0c9348907c...
1,long_short_performance.parquet,144,10076,119987d2a342188cad86dd54c143e92c0682c8af9f2578...
2,portfolio_leg_performance.parquet,288,15133,c07fc7cf6c139efcef7d20c615f06bc384ce04b6a9a78a...
3,portfolio_monthly_returns.parquet,2484,23942,c150cb1d68f2f845b79268eff59078fabd1cf0a835d79d...


Actual accessible CRSP cutoff: 2025-12-31
Complete immediate windows: 4 of 4
Manifest: 15711fbbb5102a47642ccc39fc3171bfe7658c708befd83d3b79fc673a44a680
